# Where a setting comes from

Layer 2: [Configuration](../subsystems/configuration.md). Layer 3: [config_loader](../modules/config.md).

Every run starts from a scenario id. The loader (`config/config_loader.py`) merges, in order, the base defaults
(`config/defaults/base.yaml` and `config/defaults/base/**`), the globals in `config/config.yaml`, the overlay for the
scenario's `method`, the overlay for its `targets.source`, the overlay for its `market.dispatch`, and finally the
scenario's own entry. This notebook asks the loader to explain one scenario, `Test_Market`, the default run.

In [1]:
import logging, sys, warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Snakefile").exists())
for path in (ROOT, ROOT / "config"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
warnings.filterwarnings("ignore")
logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
logging.getLogger("scripts").setLevel(logging.WARNING)

import pandas as pd
pd.set_option("display.width", 120)

In [2]:
import config_loader

explained = config_loader.explain_scenario("Test_Market")
layers = explained["layers"]
print(f"{len(layers)} files, merged in this order:")
for i, layer in enumerate(layers, 1):
    print(f"{i:3d}. {layer}")

23 files, merged in this order:
  1. config/defaults/base.yaml
  2. config/defaults/base/consumer_bill.yaml
  3. config/defaults/base/demand.yaml
  4. config/defaults/base/fleet.yaml
  5. config/defaults/base/interconnectors.yaml
  6. config/defaults/base/marginal_costs.yaml
  7. config/defaults/base/market/balancing.yaml
  8. config/defaults/base/market/wholesale.yaml
  9. config/defaults/base/market.yaml
 10. config/defaults/base/network.yaml
 11. config/defaults/base/public_ownership/acquisition.yaml
 12. config/defaults/base/public_ownership/consumer_bill.yaml
 13. config/defaults/base/public_ownership/reference_run_groups.yaml
 14. config/defaults/base/public_ownership/sensitivities.yaml
 15. config/defaults/base/public_ownership.yaml
 16. config/defaults/base/renewables.yaml
 17. config/defaults/base/solve.yaml
 18. config/defaults/base/storage.yaml
 19. config/defaults/base/support.yaml
 20. config/defaults/base/thermal.yaml
 21. config/defaults/method/gsp_block.yaml
 22. config

The defaults are split by key path: `config/defaults/base/market/balancing.yaml` holds the keys under
`market.balancing`. The overlays only appear when the scenario declares the matching axis; `Test_Market` is a FES
scenario built by GSP blocks, so `method/gsp_block.yaml` and `source/FES.yaml` are in the list.

In [3]:
values = pd.DataFrame(explained["values"], columns=["key", "value", "set by"])
print(f"{len(values)} leaf settings")
values["set by"].value_counts().rename("settings").to_frame()

2066 leaf settings


,settings
set by,
config/defaults/base/consumer_bill.yaml,430
config/defaults/base/public_ownership/reference_run_groups.yaml,234
config/defaults/base/market/balancing.yaml,208
config/defaults/base/public_ownership.yaml,179
config/defaults/base/public_ownership/sensitivities.yaml,176
config/defaults/base/public_ownership/acquisition.yaml,110
config/defaults/base/network.yaml,108
config/defaults/base/marginal_costs.yaml,104
config/defaults/base/support.yaml,101


Most settings come from the base defaults; the scenario itself sets only what makes it this scenario. The keys that
decide what kind of run it is:

In [4]:
keys = ["targets.source", "targets.pathway", "method", "network_model", "modelled_year",
        "market.enabled", "solve.solver.name", "solve_period.start", "solve_period.end"]
values.set_index("key").reindex(keys).dropna(how="all")

,value,set by
key,,
targets.source,FES,config/scenarios.yaml: scenario Test_Market
targets.pathway,HT,config/scenarios.yaml: scenario Test_Market
method,gsp_block,config/scenarios.yaml: scenario Test_Market
network_model,Reduced,config/scenarios.yaml: scenario Test_Market
modelled_year,2035,config/scenarios.yaml: scenario Test_Market
market.enabled,True,config/scenarios.yaml: scenario Test_Market
solve.solver.name,gurobi,config/defaults/base/solve.yaml
solve_period.start,2035-01-01 00:00,config/scenarios.yaml: scenario Test_Market
solve_period.end,2035-01-01 05:00,config/scenarios.yaml: scenario Test_Market


The same question from the command line, for one key:

```bash
python config/config_loader.py --explain Test_Market --path solve.solver.name
```

Two rules follow from this. Read the configuration only through the loader, never with `yaml.safe_load` on a
scenario file, because a scenario file alone is not the scenario. And when a script falls back to a default, the
fallback must equal the value in `config/defaults/base/`, because that is what every scenario that does not set the
key receives.